## Setup & Imports

In [ ]:
!pip install -q torch_lr_finder
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
import torch.nn as nn
import os
import subprocess

from torch.utils.data import DataLoader
from torch_lr_finder import LRFinder

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset

from transformers import (
    AutoConfig,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    NllbTokenizer,
    Seq2SeqTrainingArguments,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)


## Authentication & GPU Check

In [ ]:
token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)

print(f"CUDA AVAILABLE : {torch.cuda.is_available()}")

try:
    subprocess.run(["nvidia-smi", "-L"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found. No NVIDIA GPU detected.")


## Model, Tokenizer & Quantization Config

In [ ]:
model_id = "facebook/nllb-200-distilled-1.3B"

model_config = AutoConfig.from_pretrained(model_id)
model_config.tie_word_embeddings = False

tokenizer = NllbTokenizer.from_pretrained(
    model_id,
    src_lang="eng_Latn",
    tgt_lang="ibo_Latn",
    truncation_side="right",
    model_max_length=256,
    truncation = True
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32, 
)

nf4_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    config=model_config,
    quantization_config=bnb_config,
    torch_dtype = torch.float32,
)


## LoRA Config

In [ ]:
lora_config = LoraConfig(
    target_modules="all-linear",
    use_rslora=True,
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="lora_only",
    task_type="SEQ_2_SEQ_LM",
)


nf4_model = prepare_model_for_kbit_training(nf4_model)

qlora_model = get_peft_model(nf4_model, lora_config)


## Load Datasets

In [ ]:
igbo_dataset_train = load_dataset(
    "coded-by-49/langbridge_ai_v3_igbo_datasets",
    split="train",
)

igbo_dataset_validation = load_dataset(
    "coded-by-49/langbridge_ai_v3_igbo_datasets",
    data_files={
        "validation": "hf://datasets/"
                      "coded-by-49/langbridge_ai_v3_igbo_datasets/"
                      "igbo_eng_lv1_validation_256_filtered.jsonl"
    },
    split="validation",
)


# Tokenizer Function

In [ ]:
def tokenize_function(batch):
    eng_inputs = [entry["eng_Latn"] for entry in batch["translation"]]
    ibo_targets =  [entry["ibo_Latn"] for entry in batch["translation"]]

    model_inputs = tokenizer(
        text =  eng_inputs,
        max_length=256,
        truncation=True,

    )

    labels = tokenizer(
        text_target = ibo_targets ,
        max_length=256,
        truncation=True,
        
    )

    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs
    

# Maping my dataset to the tokenize function


In [ ]:
igbo_dataset_train = igbo_dataset_train.map(tokenize_function, batched=True,    remove_columns=igbo_dataset_validation.column_names,)
igbo_dataset_validation = igbo_dataset_validation.map(tokenize_function, batched=True,    remove_columns=igbo_dataset_validation.column_names,)

## Device Check

In [ ]:
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("no cuda device here")



## Data Collator & DataLoaders

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_id,
    pad_to_multiple_of=8,
    padding=True,
)

#Setting up gradient accumulation for lr finder 

desired_batch_size, real_batch_size = 24,6
accumulation_steps = desired_batch_size // real_batch_size

train_loader = DataLoader(
    igbo_dataset_train,
    batch_size=real_batch_size,
    shuffle=True,
    collate_fn=data_collator,
)

val_loader = DataLoader(
    igbo_dataset_validation,
    batch_size=real_batch_size,
    shuffle=False,
    collate_fn=data_collator,
)


In [ ]:
from torch_lr_finder import LRFinder, TrainDataLoaderIter, ValDataLoaderIter
import inspect

class NLLBTrainDataLoaderIter(TrainDataLoaderIter):
    def inputs_labels_from_batch(self, batch_data):
        inputs = {
            "input_ids": batch_data["input_ids"],
            "attention_mask": batch_data["attention_mask"],
            "labels": batch_data["labels"],
        }
        return inputs, batch_data["labels"]


class NLLBValDataLoaderIter(ValDataLoaderIter):
    def inputs_labels_from_batch(self, batch_data):
        inputs = {
            "input_ids": batch_data["input_ids"],
            "attention_mask": batch_data["attention_mask"],
            "labels": batch_data["labels"],
        }
        return inputs, batch_data["labels"]

In [ ]:
train_iter = NLLBTrainDataLoaderIter(train_loader)
val_iter = NLLBValDataLoaderIter(val_loader)

## LR Finder

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
import torch.nn as nn

class NLLBModelWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, inputs):
        return self.model(**inputs)

In [ ]:
def nllb_criterion(outputs, labels):
    return outputs.loss

In [ ]:
"""
Crash-tolerant wrapper around torch-lr-finder's range_test (built for Kaggle).

What you get
  1. progress.log          - a log line every `log_every` iterations (flushed immediately)
  2. history_partial.json  - lr/loss history checkpointed every `save_every` iterations
  3. lr_finder_plot.png    - the graph, saved as soon as the test ends (or crashes)
     result.json           - suggested_lr + full history + status
  4. tqdm bar with elapsed<remaining and live lr/loss, e.g.
     LR range test:  12%|#2   | 60/501 [01:10<08:35, 1.17s/it, lr=3.20e-06, loss=2.3101]
"""
import json
import logging
import os
import sys
import time
from datetime import timedelta

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch_lr_finder.lr_finder as _lrf_module

OUT_DIR = "/kaggle/working/lr_finder_run"   # /kaggle/working is what Kaggle keeps as output


def _hms(seconds):
    return str(timedelta(seconds=int(seconds)))


def _get_logger(path):
    logger = logging.getLogger("lr_finder")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()          # avoids duplicate lines when the cell is re-run
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")
    for h in (logging.FileHandler(path, mode="a"), logging.StreamHandler(sys.stdout)):
        h.setFormatter(fmt)
        logger.addHandler(h)
    return logger


def _atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)            # a crash mid-write can never corrupt the saved file


def run_lr_range_test(finder, num_iter, out_dir=OUT_DIR, log_every=10, save_every=25,
                      reset_after=True, **range_test_kwargs):
    """Runs finder.range_test(num_iter=..., **kwargs) with tracking + saving. Returns suggested_lr."""
    os.makedirs(out_dir, exist_ok=True)
    log = _get_logger(os.path.join(out_dir, "progress.log"))
    hist_path = os.path.join(out_dir, "history_partial.json")
    result_path = os.path.join(out_dir, "result.json")
    png_path = os.path.join(out_dir, "lr_finder_plot.png")

    meta = {"num_iter": num_iter,
            "start_lr": finder.optimizer.param_groups[0]["lr"],
            **{k: v for k, v in range_test_kwargs.items() if isinstance(v, (int, float, str))}}

    t0 = time.time()
    orig_tqdm = _lrf_module.tqdm

    def tracked_tqdm(iterable, *args, **kwargs):
        # Replaces the tqdm that range_test uses; same bar, plus logging/checkpointing.
        bar = orig_tqdm(iterable, *args, desc="LR range test", **kwargs)
        for item in bar:
            h = finder.history
            n = len(h["loss"])                       # iterations finished so far
            if n:
                bar.set_postfix(lr=f"{h['lr'][-1]:.2e}", loss=f"{h['loss'][-1]:.4f}", refresh=False)
                if n == 1 or n % log_every == 0:
                    el = time.time() - t0
                    per_it = el / n
                    gpu = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
                    # ETA is an upper bound: the test stops early if the loss diverges
                    log.info(f"[{n}/{num_iter} | {100 * n / num_iter:5.1f}%] "
                             f"lr={h['lr'][-1]:.2e} loss={h['loss'][-1]:.4f} best={finder.best_loss:.4f} "
                             f"| {per_it:.2f}s/it | elapsed {_hms(el)} | ETA {_hms(per_it * (num_iter - n))} "
                             f"| GPU {gpu:.1f}GB")
                if n % save_every == 0:
                    _atomic_json({**meta, "status": "running", "iterations_done": n, "history": h}, hist_path)
            yield item

    status, suggested_lr = "completed", None
    log.info(f"Starting range test: {meta}")
    _lrf_module.tqdm = tracked_tqdm
    try:
        finder.range_test(num_iter=num_iter, **range_test_kwargs)
    except BaseException as e:                       # includes CUDA OOM and KeyboardInterrupt
        status = f"aborted: {type(e).__name__}: {e}"
        log.error(status)
        raise
    finally:
        _lrf_module.tqdm = orig_tqdm
        h = finder.history
        n = len(h["loss"])
        if status == "completed" and n < num_iter:
            status = "stopped_early_loss_diverged"
        elapsed = _hms(time.time() - t0)

        _atomic_json({**meta, "status": status, "iterations_done": n, "history": h}, hist_path)
        if n >= 2:
            try:
                fig, ax = plt.subplots(figsize=(9, 5))
                res = finder.plot(log_lr=True, ax=ax)          # passing ax stops plot() from closing the fig
                suggested_lr = float(res[1]) if isinstance(res, tuple) else None
                fig.savefig(png_path, dpi=150, bbox_inches="tight")
                plt.show()
                plt.close(fig)
            except Exception as e:
                log.warning(f"Could not draw/save plot ({e!r}); use replot_from_json() on {hist_path}")
        _atomic_json({**meta, "status": status, "iterations_done": n, "elapsed": elapsed,
                      "suggested_lr": suggested_lr, "history": h}, result_path)
        log.info(f"Finished: {status} | {n}/{num_iter} iterations | {elapsed} | suggested_lr={suggested_lr}")
        log.info(f"Saved to {out_dir}")

        if reset_after:      # the model was trained with LRs up to end_lr, so put it back
            try:
                finder.reset()
                log.info("Model and optimizer restored to their pre-test state.")
            except Exception as e:
                log.warning(f"finder.reset() failed ({e!r}); reload your adapter/optimizer before training.")
    return suggested_lr


def replot_from_json(path, skip_start=10, skip_end=5, save_to=None):
    """Rebuild the graph from a saved history file, e.g. after the kernel died mid-test."""
    with open(path) as f:
        hist = json.load(f)["history"]
    lrs = hist["lr"][skip_start:-skip_end] if skip_end else hist["lr"][skip_start:]
    losses = hist["loss"][skip_start:-skip_end] if skip_end else hist["loss"][skip_start:]
    if len(losses) < 2:
        print("Not enough points to plot yet.")
        return
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(lrs, losses)
    ax.set_xscale("log")
    ax.set_xlabel("Learning rate")
    ax.set_ylabel("Loss")
    i = int(np.argmin(np.gradient(np.array(losses))))
    ax.scatter(lrs[i], losses[i], s=75, color="red", zorder=3, label="steepest gradient")
    ax.legend()
    print(f"Steepest-gradient LR: {lrs[i]:.2e}")
    if save_to:
        fig.savefig(save_to, dpi=150, bbox_inches="tight")
    plt.show()


# ---------------- usage (replaces your range_test + plot cells) ----------------
# suggested_lr = run_lr_range_test(
#     lr_finder,
#     num_iter=501,
#     train_loader=train_iter,
#     val_loader=val_iter,
#     end_lr=5,
#     step_mode="exp",
#     accumulation_steps=accumulation_steps,
# )
# print(f"The suggested learning rate is = {suggested_lr}")
#
# After a crash:
# replot_from_json("/kaggle/working/lr_finder_run/history_partial.json")


In [ ]:
qlora_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

wrapped_model = NLLBModelWrapper(qlora_model)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, wrapped_model.parameters()),
    lr=1e-7,
    weight_decay=1e-2,
)


lr_finder = LRFinder(
    wrapped_model,
    optimizer,
    nllb_criterion,
    device="cuda:0",
    memory_cache=False,
)


# lr_finder.range_test(
#     train_loader = train_iter,
#     val_loader= val_iter,
#     end_lr=5,
#     num_iter=501,
#     step_mode="exp",
#     accumulation_steps = accumulation_steps
# )

# print(f"logarithmic graph : \n")

# ax, suggested_lr = lr_finder.plot(log_lr=True)

# print(f"\n The suggested Learning rate is = {suggested_lr} \n")



### New range test function (claude)

In [ ]:
suggested_lr = run_lr_range_test(
    lr_finder,
    num_iter=501,
    train_loader=train_iter,
    val_loader=val_iter,
    end_lr=5,
    step_mode="exp",
    accumulation_steps=accumulation_steps,
)
print(f"The suggested learning rate is = {suggested_lr}")

In [ ]:
print(f"Linear graph : \n")
lr_finder.plot(log_lr = False)

In [ ]:
lr_finder.reset()

In [ ]:
!nvidia-smi 